# Taller 
## Fundamentos de Biología Computacional 2025-2
 
**Tema:** Evolución molecular del metabolismo del alcohol (ADH4 en homínidos)

Las alcohol deshidrogenasas (ADH) son una familia de enzimas presentes en muchos organismos, desde bacterias hasta mamíferos, cuya función principal es el metabolismo del etanol, una de las formas de alcohol.

Esta enzima ha sido especialmente relevante en la evolución de los primates, ya que su actividad frente al etanol varía significativamente entre especies (y como sabemos puede ser muy tóxica por encima de ciertas cantidads, aunque este límite es también variable). Estudios evolutivos sugieren que ADH pudo haber adquirido mayor eficiencia metabólica en linajes de homínidos terrestres (humanos, gorilas, chimpancés y bonobos) debido a su exposición a alimentos en proceso de fermentación (Carrigan MA *et al*. Hominids adapted to metabolize ethanol long before human-directed fermentation. PNAS 2015 Jan 13;112(2):458-63. doi: 10.1073/pnas.1404167111)

### 1. Recuperación de secuencias (FASTA) de ADH tipo IV


Busque la secuencia de proteína ADH en humanos (NCBI ID NP_000663.1) y descárguela en el directorio del examen.

In [ ]:
hide-input

import numpy as np
import matplotlib.pyplot as plt
plt.ion()

data = np.random.randn(2, 100)
fig, ax = plt.subplots()
ax.scatter(*data, c=data[1], s=100*np.abs(data[0]));

In [ ]:
#!/usr/local/bin/

# Script por Laura Salazar Jaramillo
# Busca la secuencia de proteinas del ADH en humanos


from Bio import Entrez
from Bio import SeqIO

Entrez.email = "lsalazarj@eafit.edu.co"

with Entrez.efetch(
    db="protein", rettype="fasta", retmode="text", id="NP_000663.1") as handle:
    seq_record = SeqIO.read(handle, "fasta")

print(seq_record)
SeqIO.write(seq_record, "adh_prot_human.fa", "fasta")
```


### 2. Búsqueda de homólogos primates
Utilice la secuencia de *Homo sapiens* para buscar los **homólogos** en, por lo menos otros 5 géneros de primates usando BLAST. Explique su criterio para definir esas secuencias como homólogas.

| Especie | Accession Number | E-value | Per Iden |
|--------|------------------| ------- | ------- |
| Human | NP_000664.3 | | |
| |  | ||
| |  | ||
| |  | ||
| |  | ||
| |  | ||

Descargue las secuencias en un archivo fasta junto con la de referencia

In [ ]:
#!/usr/local/bin/

# Script por Laura Salazar Jaramillo
# Descarga las secuencias de proteinas de ADH de primates


from Bio import Entrez
from Bio import SeqIO

Entrez.email = "lsalazarj@eafit.edu.co"

# Diccionario especies-accesiones
accessions = {
     "Marmoset (Callithrix jacchus)": "XP_002745608.1",
    "Gibbon (Nomascus leucogenys)": "XP_003257556.2",
    "Orangutan (Pongo abelii)": "PNJ35320.1",
    "Bonobo (Pan paniscus)": "XP_063459910.1",
    "Chimpanzee (Pan troglodytes)": "PNI56599.1",
    "Gorilla (Gorilla gorilla)": "XP_004039198.5",
    "Human (Homo sapiens)": "NP_000663.1",
    
}

# Output file
output_file = "ADH_primates.fasta"

# Write sequences to FASTA
with open(output_file, "w") as out_f:
    for species, acc in accessions.items():
        try:
            print(f"Fetching {species}: {acc}")
            handle = Entrez.efetch(db="protein", id=acc, rettype="fasta", retmode="text")
            seq_record = handle.read()
            handle.close()
            out_f.write(seq_record)
        except Exception as e:
            print(f"Failed to fetch {acc} for {species}: {e}")

print(f"\n All available sequences written to {output_file}")


### 3. Alineamiento y árbol filogenético

Realice un alineamiento y un árbol filogenético justificando los parámetros que escoge (matriz de distancia y agrupamiento de ramas). Qué tan similar es la proteína entre las especies? 

In [ ]:
conda activate filogenia
clustalw -infile=ADH_primates.txt -type=protein -output=phylip

También puede utilizar la herramienta online: https://www.ebi.ac.uk/Tools/msa/clustalo/

O desde R:

In [ ]:
# R

library(Biostrings)
library(ape)
library(phangorn)
library(msa)   # Para alinear las secuencias (multiple sequence aln)

# Lees proteínas homólogas
adh_prot <- readAAStringSet("ADH_primates.fasta",
  format = "fasta")

# Alinear
alignment_prot <- msa(adh_prot, method = "ClustalW")
alignment_prot_bin <- as.AAbin(alignment_prot)
adh_prot_phy <- phyDat(alignment_prot_bin, type = "AA")
# Modelo JTT (Proteína)
dist_prot <- dist.ml(adh_prot_phy, model = "JTT")

#Construir el árbol filogenético
tree_nj_prot <- NJ(dist_prot)

#Visualizar
plot(tree_nj_prot)

adh_phy_dat <- as.phyDat(adh)
adh_dist <- dist.ml(adh_phy_dat)
adh_NJ <- NJ(adh_dist)
plot(adh_NJ)


### 4. Dominios funcionales
Contraste las secuencias de proteína de ADH de humano y de un primate mas basal, *Tarsius syrichta* (primero debes buscar la secuencia homóloga, si esta especie no está incluida en el grupo de homólogos) en términos de 
a) longitud de la proteína (vía script de R o python) 
b) frecuencia de los diferentes amino ácidos (vía script de R o python)
c) dominios presentes en las proteínas (utilizando InterPro)
¿Qué tan diferentes son las proteínas?
¿Hay diferencias funcionales esperadas basadas en esta estructura?

### 5. Estructura de proteínas
Utilice AlphaFold para comparar la predicción de proteínas de ambas especies

Notas: Recuerde que los archivos NO se deben nombrar con espacios